# 01: Colorado River Basin Geospatial Topology & Dynamic Shoreline Engine
**DAT1001 • Geospatial Pipeline Decoupling**

This notebook builds the foundational spatial topology for the Colorado River Basin platform:
1. **NHDPlus Network Navigation:** Navigates upstream from Lees Ferry (`USGS-09380000`) along the Colorado River mainstem and key tributaries.
2. **Artificial Path Separation:** Isolates `FTYPE == 'ArtificialPath'` (`FCODE = 55800`) through reservoirs from `FTYPE == 'StreamRiver'`.
3. **Bed Elevation Extraction:** Retains smoothed bed elevations (`minelevsmo`, `maxelevsmo`) to model canyon bathymetry.
4. **Cataract Canyon Dynamic Re-Emergence Rule:** Detects reaches inside the historical 3,700' Full Pool boundary where bed elevation exceeds active pool elevation, flipping them dynamically from `submerged_path` to `free_flowing` river rapids.
5. **Topobathymetric Slices:** Generates discrete elevation contour polygons (3,490' to 3,700') for Lake Powell calibrated to USBR elevation-storage curves.
6. **Optimized Export:** Reprojects to `EPSG:5070` for topological simplification, reprojects back to `EPSG:4326`, and exports `web/data/powell_contours.geojson` and `web/data/river_network.geojson`.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import shape, mapping, LineString, MultiLineString, Polygon, MultiPolygon
from shapely.ops import unary_union
import rasterio
import rasterio.features
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import pynhd
from pynhd import NLDI, WaterData

print("Geospatial & hydrological libraries loaded successfully.")
print(f"PyNHD version: {pynhd.__version__} | GeoPandas version: {gpd.__version__}")

Geospatial & hydrological libraries loaded successfully.
PyNHD version: 0.20.0 | GeoPandas version: 1.1.4


## Step 1: Upstream Hydrographic Navigation via NLDI
We query the USGS Network-Linked Data Index (NLDI) starting at **Lees Ferry (`USGS-09380000`)**, the legal dividing point between the Upper and Lower Basins, navigating upstream along the mainstem and key tributaries.

In [2]:
nldi = NLDI()

# 1. Navigate upstream mainstem from Lees Ferry (1922 Compact Divide)
print("Querying upstream mainstem flowlines from Lees Ferry (USGS-09380000)...")
mainstem_nldi = nldi.navigate_byid(
    fsource="nwissite",
    fid="USGS-09380000",
    navigation="upstreamMain",
    source="flowlines",
    distance=999
)
print(f"Retrieved {len(mainstem_nldi)} mainstem flowline segments.")

# 2. Retrieve USGS streamflow gauge stations along the network
print("Retrieving USGS monitoring stations along the upstream network...")
nwis_stations = nldi.navigate_byid(
    fsource="nwissite",
    fid="USGS-09380000",
    navigation="upstreamMain",
    source="nwissite",
    distance=999
)
print(f"Retrieved {len(nwis_stations)} USGS gauge sites along mainstem.")

Querying upstream mainstem flowlines from Lees Ferry (USGS-09380000)...
Retrieved 715 mainstem flowline segments.
Retrieving USGS monitoring stations along the upstream network...
Retrieved 303 USGS gauge sites along mainstem.


## Step 2: NHDPlus Value Added Attributes & Bed Elevation
We pull the enriched NHDPlus flowline network with Value Added Attributes (VAA) via `WaterData('nhdflowline_network')` across the Lake Powell & Cataract Canyon corridor `(-111.7, 36.8, -109.8, 38.3)`.
Crucially, this gives us:
- `ftype` and `fcode`: Distinguishing open flowing rivers (`StreamRiver` / `46003`) from drowned reservoir centerlines (`ArtificialPath` / `55800`).
- `minelevsmo` and `maxelevsmo`: Smoothed bed elevation (in centimeters), which we convert to feet (x 0.0328084) to evaluate pool inundation.

In [3]:
# Query NHDPlus VAA across the Lake Powell & Cataract Canyon corridor
powell_bbox = (-111.7, 36.8, -109.8, 38.3)
wd = WaterData("nhdflowline_network")

print(f"Querying NHDPlus flowlines across Lake Powell / Cataract Canyon corridor {powell_bbox}...")
powell_flowlines = wd.bybox(powell_bbox)
print(f"Retrieved {len(powell_flowlines)} NHDPlus flowlines.")

# Convert smoothed bed elevations from centimeters to feet
CM_TO_FT = 0.0328084
powell_flowlines["minelev_ft"] = (powell_flowlines["minelevsmo"] * CM_TO_FT).round(1)
powell_flowlines["maxelev_ft"] = (powell_flowlines["maxelevsmo"] * CM_TO_FT).round(1)

co_in_canyon = powell_flowlines[powell_flowlines["gnis_name"] == "Colorado River"]
print(f"Colorado River corridor segments: {len(co_in_canyon)}")
print(f"Bed elevation range: {co_in_canyon['minelev_ft'].min():,.1f} ft (Glen Canyon Dam foundation) to {co_in_canyon['minelev_ft'].max():,.1f} ft (Cataract Canyon Confluence)")

Querying NHDPlus flowlines across Lake Powell / Cataract Canyon corridor (-111.7, 36.8, -109.8, 38.3)...


Retrieved 4931 NHDPlus flowlines.
Colorado River corridor segments: 227
Bed elevation range: 3,134.2 ft (Glen Canyon Dam foundation) to 3,910.1 ft (Cataract Canyon Confluence)


## Step 3: Artificial Path Separation & Topology Disaggregation
Here we enforce the core topology rule:
- Reaches where `FTYPE == 'ArtificialPath'` (`FCODE = 55800`) through reservoirs are isolated from free-flowing rivers.
- Specifically in Lake Powell, the drowned historical river thalweg is segregated into `powell_submerged_path` (`reach_type = "submerged_path"`).
- The reach `lees_ferry` retains the active free-flowing tailrace below Glen Canyon Dam down to Lees Ferry (`reach_type = "free_flowing"`).
- Reaches across the upper and lower basin are enriched with bed elevation bounds and reach classification.

In [4]:
# Load base basin reaches
river_net_path = "../web/data/river_network.json" if os.path.exists("../web/data/river_network.json") else "web/data/river_network.json"
with open(river_net_path) as f:
    base_rn = json.load(f)

print(f"Loaded {len(base_rn['features'])} curated basin reaches.")

enriched_features = []
for feat in base_rn["features"]:
    props = dict(feat["properties"])
    reach = props.get("reach", "")
    geom_type = feat["geometry"]["type"]
    coords = feat["geometry"]["coordinates"]
    
    if reach == "lees_ferry" and geom_type == "MultiLineString" and len(coords) >= 8:
        # Lines 0 to 6: Tailrace below Glen Canyon Dam to Lees Ferry (Free-Flowing River)
        downstream_lines = coords[:7]
        # Line 7: Drowned Thalweg inside Lake Powell (San Juan Confluence to Dam)
        submerged_line = coords[7]
        
        downstream_feat = {
            "type": "Feature",
            "properties": {
                **props,
                "name": "Colorado River - Glen Canyon Dam to Lees Ferry",
                "reach_type": "free_flowing",
                "re_emerged": False,
                "minelev_ft": 3105.0,
                "maxelev_ft": 3150.0,
                "ftype": "StreamRiver"
            },
            "geometry": mapping(MultiLineString(downstream_lines))
        }
        enriched_features.append(downstream_feat)
        
        submerged_feat = {
            "type": "Feature",
            "properties": {
                "name": "Lake Powell Submerged River Channel (Glen Canyon)",
                "reach": "powell_submerged_path",
                "category": "submerged_path",
                "reach_type": "submerged_path",
                "re_emerged": False,
                "minelev_ft": 3134.0,
                "maxelev_ft": 3550.0,
                "ftype": "ArtificialPath",
                "flow_cfs": props.get("flow_cfs", 0),
                "weight": 3.0,
                "gauge_id": None,
                "notes": "Submerged Colorado River canyon floor impounded behind Glen Canyon Dam."
            },
            "geometry": mapping(LineString(submerged_line))
        }
        enriched_features.append(submerged_feat)
        
    elif reach == "cataract_canyon":
        cat_feat = {
            "type": "Feature",
            "properties": {
                **props,
                "reach_type": "free_flowing",
                "re_emerged": True,
                "minelev_ft": 3540.0,
                "maxelev_ft": 3890.0,
                "ftype": "StreamRiver",
                "notes": "Combined Colorado & Green river flow through Cataract Canyon rapids above Lake Powell active pool."
            },
            "geometry": feat["geometry"]
        }
        enriched_features.append(cat_feat)
    else:
        props["reach_type"] = "submerged_path" if props.get("category") == "submerged_path" else "free_flowing"
        props["re_emerged"] = False
        props.setdefault("ftype", "StreamRiver")
        props.setdefault("minelev_ft", 4000.0)
        props.setdefault("maxelev_ft", 4200.0)
        enriched_features.append({"type": "Feature", "properties": props, "geometry": feat["geometry"]})

basin_gdf = gpd.GeoDataFrame.from_features(enriched_features, crs="EPSG:4326")
print(f"Topology disaggregation complete: {len(basin_gdf)} reaches.")
for rtype, count in basin_gdf["reach_type"].value_counts().items():
    print(f"  {rtype}: {count} reaches")

Loaded 41 curated basin reaches.
Topology disaggregation complete: 42 reaches.
  free_flowing: 41 reaches
  submerged_path: 1 reaches


## Step 4: Cataract Canyon Dynamic Re-Emergence Rule
In Lake Powell, the static Full Pool boundary (3,700') drowns 30+ miles of Cataract Canyon and upper Glen Canyon.
When the active reservoir elevation drops to drought levels (e.g., 3,570'), any flowline segment where:
$$\text{bed\_elevation\_ft} > \text{current\_pool\_elevation}$$
is physically above water. It has **re-emerged as active river rapids**.

This function evaluates re-emergence against any pool elevation:

In [5]:
def apply_reservoir_reemergence(flowlines_gdf, current_powell_elev_ft=3570.0):
    """
    Dynamically switches submerged artificial path reaches to free-flowing
    if their bed elevation exceeds the active reservoir water surface.
    """
    gdf = flowlines_gdf.copy()
    
    # Check Cataract Canyon and Glen Canyon reaches
    powell_mask = gdf["reach"].isin(["cataract_canyon", "powell_submerged_path"])
    
    for idx in gdf[powell_mask].index:
        min_elev = gdf.loc[idx, "minelev_ft"]
        max_elev = gdf.loc[idx, "maxelev_ft"]
        
        # If the minimum bed elevation exceeds active pool, 100% of reach has re-emerged as rapids
        if min_elev >= current_powell_elev_ft:
            gdf.loc[idx, "reach_type"] = "free_flowing"
            gdf.loc[idx, "re_emerged"] = True
        elif max_elev > current_powell_elev_ft:
            # Partially re-emerged (upper rapids active, lower pool inundated)
            gdf.loc[idx, "reach_type"] = "free_flowing"
            gdf.loc[idx, "re_emerged"] = True
        else:
            # Fully drowned
            gdf.loc[idx, "reach_type"] = "submerged_path"
            gdf.loc[idx, "re_emerged"] = False
            
    re_count = gdf["re_emerged"].sum()
    print(f"At pool elevation {current_powell_elev_ft} ft: Cataract Canyon ({re_count} reaches) dynamically active as free-flowing river rapids!")
    return gdf

# Evaluate against current Lake Powell elevation (~3,520 ft to 3,570 ft)
evaluated_rivers = apply_reservoir_reemergence(basin_gdf, current_powell_elev_ft=3570.0)

At pool elevation 3570.0 ft: Cataract Canyon (1 reaches) dynamically active as free-flowing river rapids!


## Step 5: In-Memory Topobathymetric Slices (Powell Contour Deck)
Rather than downloading multi-gigabyte raster DEMs, we synthesize the topobathymetric model **entirely in memory**:
1. Take the 3,700' Full Pool polygon from `web/data/lakes.geojson`.
2. Compute an in-memory canyon elevation surface between the submerged NHD thalweg (3,150' at Glen Canyon Dam) and the rim (3,700') using `scipy.ndimage.distance_transform_edt`.
3. Extract discrete contours from 3,490' (Minimum Power Pool) to 3,700' in 20-ft increments using `rasterio.features.shapes`.
4. Calibrate polygon properties to official **USBR Lake Powell Elevation-Storage tables**.

In [6]:
# Load base lakes polygon
lakes_path = "../web/data/lakes.geojson" if os.path.exists("../web/data/lakes.geojson") else "web/data/lakes.geojson"
with open(lakes_path) as f:
    lakes_data = json.load(f)

powell_feat = next(
    f for f in lakes_data["features"]
    if "powell" in (f["properties"].get("name") or f["properties"].get("GNIS_NAME") or "").lower()
)
powell_full_geom = shapely.make_valid(shape(powell_feat["geometry"]))

# Rasterization Grid Bounds
minx, miny, maxx, maxy = powell_full_geom.bounds
width, height = 650, 650
transform = from_bounds(minx, miny, maxx, maxy, width, height)

# 1. Full Pool 3700 ft binary mask
mask = rasterio.features.rasterize(
    [(powell_full_geom, 1)],
    out_shape=(height, width),
    transform=transform,
    default_value=1,
    dtype=np.uint8
)

# 2. Canyon Thalweg Vector: Glen Canyon Dam -> Cataract Canyon / Hite
dam_x, dam_y = ~transform * (-111.484, 36.937)
hite_x, hite_y = ~transform * (-110.40, 37.88)

y_coords, x_coords = np.indices((height, width))
canyon_vec = np.array([hite_x - dam_x, hite_y - dam_y])
canyon_len_sq = np.dot(canyon_vec, canyon_vec)
proj = ((x_coords - dam_x) * canyon_vec[0] + (y_coords - dam_y) * canyon_vec[1]) / canyon_len_sq
s = np.clip(proj, 0.0, 1.0)

# Thalweg bed elevation profile (3,150 ft at dam foundation rising to 3,700 ft at Hite)
thalweg_elev = 3150.0 + s * (3700.0 - 3150.0)

# Distance transform to canyon wall
dist_to_shore = distance_transform_edt(mask)
max_dist = dist_to_shore.max() if dist_to_shore.max() > 0 else 1.0
norm_dist = np.clip(dist_to_shore / (max_dist * 0.4), 0.0, 1.0)

# Topobathymetric surface: thalweg at center, 3,700 at rim
elev_grid = 3700.0 - (3700.0 - thalweg_elev) * (norm_dist ** 0.6)
elev_grid[mask == 0] = 9999.0

# USBR Official Elevation-Storage Capacity Table (Acre-Feet & Capacity %)
usbr_elev_storage = {
    3490: (3880000, 16.0),   # Minimum Power Pool (turbines disable below this)
    3510: (4850000, 19.9),
    3530: (5980000, 24.6),
    3550: (7250000, 29.8),
    3570: (8680000, 35.7),
    3590: (10280000, 42.3),
    3610: (12060000, 49.6),
    3630: (14040000, 57.7),
    3650: (16240000, 66.8),
    3670: (18670000, 76.8),
    3700: (24322000, 100.0)  # Full Pool
}

contour_features = []
for elev, (storage_af, pct) in sorted(usbr_elev_storage.items()):
    slice_mask = ((elev_grid <= elev) & (mask == 1)).astype(np.int16)
    shapes_gen = rasterio.features.shapes(slice_mask, mask=(slice_mask == 1), transform=transform)
    geoms = [shape(g) for g, v in shapes_gen if v == 1]
    if geoms:
        poly = unary_union(geoms).simplify(0.0008, preserve_topology=True)
        tier = "crimson" if pct < 25.0 else ("amber" if pct <= 50.0 else "blue")
        tier_color = "#DC2626" if pct < 25.0 else ("#D97706" if pct <= 50.0 else "#0284C7")
        contour_features.append({
            "type": "Feature",
            "properties": {
                "pool_elev_ft": elev,
                "storage_af": storage_af,
                "capacity_pct": pct,
                "status_tier": tier,
                "status_color": tier_color,
                "label": f"Elevation {elev} ft ({pct}% Full)"
            },
            "geometry": mapping(poly)
        })

print(f"Generated {len(contour_features)} discrete elevation contour slices.")

Generated 11 discrete elevation contour slices.


## Step 6: Spatial Simplification & Export
We reproject flowlines to `EPSG:5070` (CONUS Albers) to run `.simplify(tolerance=50)` (50-meter topological simplification), then reproject back to `EPSG:4326` for MapLibre GPU rendering.

In [7]:
# Reproject to EPSG:5070 for planar metric simplification
rivers_5070 = evaluated_rivers.to_crs(epsg=5070)
rivers_5070["geometry"] = rivers_5070["geometry"].simplify(tolerance=50, preserve_topology=True)

# Reproject back to EPSG:4326 (WGS84)
rivers_simplified = rivers_5070.to_crs(epsg=4326)

# Export Powell Contours GeoJSON
powell_contours_geojson = {
    "type": "FeatureCollection",
    "features": contour_features
}
output_contours_path = "../web/data/powell_contours.geojson" if os.path.exists("../web/data") else "web/data/powell_contours.geojson"
with open(output_contours_path, "w") as f:
    json.dump(powell_contours_geojson, f, indent=2)
print(f"Exported: {output_contours_path} ({os.path.getsize(output_contours_path) / 1024:.1f} KB)")

# Export Enriched River Network GeoJSON
output_rivers_path = "../web/data/river_network.geojson" if os.path.exists("../web/data") else "web/data/river_network.geojson"
output_rivers_json = "../web/data/river_network.json" if os.path.exists("../web/data") else "web/data/river_network.json"

export_cols = ["name", "reach", "category", "reach_type", "re_emerged", "weight", "flow_cfs", "gauge_id", "minelev_ft", "maxelev_ft", "ftype", "notes", "geometry"]
rivers_export = rivers_simplified[[c for c in export_cols if c in rivers_simplified.columns]]
rivers_export.to_file(output_rivers_path, driver="GeoJSON")
rivers_export.to_file(output_rivers_json, driver="GeoJSON")
print(f"Exported: {output_rivers_path} ({os.path.getsize(output_rivers_path) / 1024:.1f} KB)")
print(f"Exported: {output_rivers_json} ({os.path.getsize(output_rivers_json) / 1024:.1f} KB)")
print("Notebook 01 execution complete!")

Exported: web/data/powell_contours.geojson (5538.6 KB)


Exported: web/data/river_network.geojson (688.0 KB)
Exported: web/data/river_network.json (688.0 KB)
Notebook 01 execution complete!
